# French analogy: run experiments

Loads `all_cross_instances.parquet` (built by `prepare_inputs_french.ipynb`) and runs analogy experiments against MLS French wav2vec2 hidden states. Each experiment specifies a `group_by` / `base_query` / `inflected_query` over the all_cross_instances DataFrame; `analogy.run_experiment_equiv_level` does the lift.

Experiment set (see `notes/analogy-french-design.md`):
- `basic_by_suffix`: per phonological-suffix accuracy on true-friend rows.
- `by_direction`: per (suffix, base_cell, derived_cell) accuracy — the homophony test.
- `false_friend_by_suffix`: control. Does the (true_friend) analogy vector transfer to false_friend rows sharing the surface suffix? Expect: no.
- `cross_dir-*`: programmatically generated. For each suffix with ≥2 well-attested directions, transfer accuracy from one direction to another. Tests whether the model encodes the morphological direction independently of phonological suffix.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.analysis import analogy
from src.analysis.state_space import (
    StateSpaceAnalysisSpec, prepare_state_trajectory, flatten_trajectory)
from src.datasets.speech_equivalence import SpeechHiddenStateDataset

In [ ]:
dataset_name = "mls_french-dev"
base_model = "w2v2_pc_fr_8"

analogy_inputs_dir = f"outputs/analogy/inputs/{dataset_name}/{base_model}"
all_cross_instances_path = f"{analogy_inputs_dir}/all_cross_instances.parquet"
state_space_spec_path = f"{analogy_inputs_dir}/state_space_spec.h5"

# Use w2v2 hidden states directly (no trained probe).
embeddings_path = "ID"
hidden_states_path = f"outputs/hidden_states/{base_model}/{dataset_name}.h5"

output_dir = f"outputs/analogy/runs/{dataset_name}/{base_model}"

device = "cpu"
metric = "cosine"
num_samples = 200
seed = 42

# A direction is eligible for the programmatic cross-direction expansion if it
# has at least this many instance pairs in all_cross_instances.
min_instances_per_direction = 5

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
np.random.seed(seed)

## Load representations

In [ ]:
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_spec_path, "word")
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations = np.load(f)
assert state_space_spec.is_compatible_with(model_representations)
print(f"state_space_spec: {len(state_space_spec.labels)} labels, "
      f"representations shape: {model_representations.shape}")

In [ ]:
trajectory_agg = prepare_state_trajectory(
    model_representations, state_space_spec,
    agg_fn_spec="mean", agg_fn_dimension=1)
agg, agg_src = flatten_trajectory(trajectory_agg)
print(f"agg: {agg.shape}, agg_src: {agg_src.shape}")

## Load all_cross_instances + homophone map

In [ ]:
all_cross_instances = pd.read_parquet(all_cross_instances_path)
print(f"all_cross_instances: {len(all_cross_instances)} rows")
print(all_cross_instances.groupby(["type", "suffix_ipa"]).size())

In [ ]:
# Homophone map: label_idx → set of label_idxs sharing some phonemic form. Used by
# analogy.run_experiment_equiv_level's include_idxs_in_predictions to count
# homophone predictions as correct. (French homophony is rife: dirent/dirent[FF],
# parle/parlent in continuous speech, etc.)
from collections import defaultdict
cuts_df = state_space_spec.cuts.xs("phoneme", level="level")
cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)
pron2label = defaultdict(set)
for label, rows in cut_phonemic_forms.groupby("label"):
    for pron in set(rows):
        pron2label[pron].add(label)
homophone_label_map = {label: {h for pron in set(cut_phonemic_forms.loc[label])
                               for h in pron2label[pron]}
                       for label in state_space_spec.labels}
homophone_map = {state_space_spec.labels.index(label):
                 {state_space_spec.labels.index(h) for h in homs}
                 for label, homs in homophone_label_map.items()}
print(f"homophone_map: {len(homophone_map)} entries; "
      f"avg homophones per label: {np.mean([len(v) for v in homophone_map.values()]):.2f}")

## Define experiments

In [ ]:
experiments = {
    "basic_by_suffix": {
        "group_by": ["suffix_ipa"],
        "all_query": "type == 'true_friend' and not exclude_main",
    },
    "by_direction": {
        "group_by": ["suffix_ipa", "base_cell", "derived_cell"],
        "all_query": "type == 'true_friend' and not exclude_main",
    },
    "false_friend_by_suffix": {
        "group_by": ["suffix_ipa"],
        "base_query": "type == 'true_friend'",
        "inflected_query": "type == 'false_friend'",
    },
}

In [ ]:
# Programmatic cross-direction expansion: for each suffix with ≥2 well-attested
# directions, test transfer between every pair of directions. Tests whether the
# encoded "morphological vector" generalizes across cells with shared surface
# suffix — the central homophony question.
well_attested = (
    all_cross_instances
    .query("type == 'true_friend' and not exclude_main")
    .groupby(["suffix_ipa", "base_cell", "derived_cell"])
    .size()
    .loc[lambda s: s >= min_instances_per_direction]
    .reset_index())
print(f"well-attested (suffix, direction) tuples: {len(well_attested)}")

for suffix, group in well_attested.groupby("suffix_ipa"):
    directions = list(zip(group.base_cell, group.derived_cell))
    for (b1, d1), (b2, d2) in itertools.combinations(directions, 2):
        name = f"cross_dir-{suffix}-{b1}-{d1}-to-{b2}-{d2}"
        experiments[name] = {
            "base_query": (f"suffix_ipa == '{suffix}' and base_cell == '{b1}' "
                           f"and derived_cell == '{d1}'"),
            "inflected_query": (f"suffix_ipa == '{suffix}' and base_cell == '{b2}' "
                                f"and derived_cell == '{d2}'"),
        }
print(f"total experiments (including programmatic cross-dir): {len(experiments)}")

## Run

In [ ]:
experiment_results = pd.concat({
    name: analogy.run_experiment_equiv_level(
        name, config, state_space_spec, all_cross_instances,
        agg, agg_src,
        num_samples=num_samples,
        seed=seed,
        device=device,
        include_idxs_in_predictions=homophone_map)
    for name, config in tqdm(experiments.items(), unit="experiment")
}, names=["experiment"])
experiment_results["correct"] = (
    experiment_results.predicted_label == experiment_results.gt_label)
experiment_results

## Save

In [ ]:
experiment_results.to_csv(f"{output_dir}/experiment_results.csv")
print(f"Saved to {output_dir}/experiment_results.csv ({len(experiment_results)} rows)")